# 实践 5.2  QTG-seq输入格式与平滑教学示例

《统计遗传与智能育种学》第五章 · 电子资源 `05b_qtgseq_smoothlod.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：05b_qtgseq_smoothlod.ipynb。实践按照QTG-seq的六列输入结构生成虚拟混池数据，演示单点负对数P值及其滑动均值，单独导出正式软件所需六列计数；Notebook的Markdown单元保留QTG_Seq.sh与R函数调用模板，便于安装真实软件后替换模拟输入。QTG-seq官方仓库为caulilin/QTG_Seq。

QTG-seq命令接口示例：./QTG_Seq.sh [VCF] [GFF] [Cov Threshold] [Win Size] <EuclideanDist>  
R函数接口示例：LOD(dir="D:/users", filegen="gen.csv", width=3375000, DrawPlot=TRUE, chrom=c(7,8), col=c("blue","red"), Plotformat1="tiff", Resolution="High")  
注意：书中保留接口思想；实际参数名、依赖和版本以项目README为准。

## 输入、完整代码、实际输出与解释

输入与可复现条件：按gen.csv六列结构生成虚拟混池数据；SEED=502；运行写出outputs_05b/gen.csv和Final_result.csv。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED=502
rng=np.random.default_rng(SEED)
print('Python',sys.version.split()[0])
print('Platform',platform.platform())
print('numpy',np.__version__,'pandas',pd.__version__,'scikit-learn',sklearn.__version__)
print('SEED',SEED)


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 502


In [2]:
from scipy.stats import norm
n=700; chrom=np.ones(n,int); pos=np.arange(n)*12000; qtl=4_200_000; p0=rng.uniform(.2,.8,n); d=.25*np.exp(-((pos-qtl)/550000)**2); pa=np.clip(p0+d,0.02,.98); pb=np.clip(p0-d,0.02,.98); da=rng.poisson(60,n)+1; db=rng.poisson(60,n)+1; aa=rng.binomial(da,pa); ab=rng.binomial(db,pb)
gen=pd.DataFrame({'Chr':chrom,'Pos':pos,'RefLow':db-ab,'AltLow':ab,'RefHigh':da-aa,'AltHigh':aa})
fa=aa/da; fb=ab/db; diff=fa-fb; se=np.sqrt(np.maximum(fa*(1-fa)/da+fb*(1-fb)/db,1e-9)); z=diff/se; pval=2*norm.sf(np.abs(z)); lod=-np.log10(np.maximum(pval,1e-300)); gen['NegLog10P']=lod; gen['SmoothNegLog10P']=pd.Series(lod).rolling(31,center=True,min_periods=5).mean(); final=gen[['Chr','Pos','NegLog10P','SmoothNegLog10P']].dropna(); print(final.nlargest(8,'SmoothNegLog10P').round(3).to_string(index=False))

from pathlib import Path
out=Path('outputs_05b'); out.mkdir(exist_ok=True); gen.iloc[:,:6].to_csv(out/'gen.csv',index=False); final.to_csv(out/'Final_result.csv',index=False); peak=int(final.loc[final.SmoothNegLog10P.idxmax(),'Pos']); print('peak',peak,'error bp',abs(peak-qtl)); print('wrote',out/'gen.csv',out/'Final_result.csv')


 Chr     Pos  NegLog10P  SmoothNegLog10P
   1 4176000      5.878           11.319
   1 4200000     21.071           11.201
   1 4164000     12.757           11.173
   1 4188000      8.107           11.157
   1 4212000     14.848           11.012
   1 4140000      8.401           10.946
   1 4224000      8.090           10.827
   1 4152000     10.025           10.634
peak 4176000 error bp 24000
wrote outputs_05b/gen.csv outputs_05b/Final_result.csv


结果解释：平滑负对数P值最高位置为4,176,000 bp，距模拟真值24 kb。这里用简化示例演示输入—平滑—输出逻辑，便于检查字段和窗口含义；该实现不等同于QTG-seq软件内部算法，也不能替代正式显著性分析。

可修改练习：改变平滑窗口宽度和覆盖深度，记录峰位与峰宽；再核对六列计数顺序，将导出的gen.csv交给正式SmoothLOD版本，比较输出字段和差异并记录软件版本。